# Assemble multiome data in SnapATAC2

In [ ]:
%reload_ext autoreload
%autoreload 0

### Load packages

In [ ]:
import sys
sys.path.append('/projects/site/pred/ihb-g-deco/USERS/adaml9/sc-ops')

from functools import partial
from pathlib import Path

from sc_ops.settings import *
from sc_ops.settings._atac import *

### Load settings

In [ ]:
# Load settings
settings = Dynaconf(settings_files=["/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/global_analysis/multiome/standalone_v2/settings.toml"])

# Specify output directory for all results
output_dir = Path(settings.IO.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

### Load genome

In [ ]:
from sc_ops.io._genome import load_hg38_genome

# Load genome and annotation
genome = load_hg38_genome()

### Load fragment files

In [ ]:
# List of all fragment files
paths = [
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/parse_data/atac/ITBOGE013_YBP2_14_Multiome_1.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/parse_data/atac/ITBOGE013_YBP2_14_Multiome_2.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/parse_data/atac/ITBOGE013_YBP2_14_Multiome_1.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/parse_data/atac/ITBOGE013_YBP2_14_Multiome_2.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/multiome_data/atac/ITBOGE001_BMP2-4_rep1.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/multiome_data/atac/ITBOGE001_BMP2-4_rep2.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/multiome_data/atac/ITBOGE001_Fujii.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/parse_data/atac/ITBOGE013_YBP2_14_Multiome_1.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/parse_data/atac/ITBOGE013_YBP2_14_Multiome_2.tsv.gz",
    "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/multiome_data/atac/ITBOGE001_Fujii.tsv.gz",
]
# Create a dictionary mapping sample names to file paths
files = {Path(x).stem.replace('.tsv', ''): x for x in paths}

### Generate anndata objects for ATAC data

In [ ]:
%autoreload 2

from sc_ops.hpc._submitit import create_executor
from sc_ops.preprocessing._fragments import import_fragments

In [ ]:
# Create a submitit executor with specified parameters
executor = create_executor(folder="submitit_logs", cpus_per_task=1, timeout_min=23*60, mem_gb=200, qos="1d", slurm_partition="batch_cpu")
# Create partial for the import_fragments function with fixed output_dir and genome arguments
run_job = partial(import_fragments, output_dir=output_dir, genome=genome)

In [ ]:
# Submit the job
job = executor.submit(run_job, files)
print("Submitted import job:", job.job_id)

### Load anndata objects

In [ ]:
# Get adata files
adatas = {name: output_dir / (name + '.atac.raw.h5ad') for name in files.keys()}

In [ ]:
# Get adata files
files = [output_dir / (name + '.atac.raw.h5ad') for name in files.keys()]
files = [files[0], files[1], files[4]] 
# Get basenames 
basenames = [f.stem.replace('.atac.raw', '') for f in files]
# Read in anndata objects
adatas = [sc.read(f) for f in files]
# Update barcodes to include sample name
for i, adt in enumerate(adatas):
    adt.obs_names = [f"{basenames[i]}#{bc}" for bc in adt.obs_names]

### Load integrated object

In [ ]:
# Load integrated multiome object
adata_rna = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/10X.multiome.processed.controls.integrated.h5ad")

# Remove color information to avoid issues with downstream plotting
del adata_rna.uns["final_annotation_colors"]

In [ ]:
adatas

In [ ]:
### Subset each 

### Import peaks called from ArchR

In [ ]:
## Requantify using SnapATAC2

In [ ]:
peak_mat = snap.pp.make_peak_matrix(data, use_rep=peaks['Peaks'])

In [ ]:
1+1

In [ ]:
# Combine all adata objects into one
adata_atac = adatas[0].concatenate(adatas[1:], batch_key="sample", index_unique=None)

In [ ]:
adata_rna

In [ ]:
sc.pl.embedding(adata_rna, basis="X_umap_harmony", color="final_annotation")

### Load matched EEC multiome object

In [ ]:
adata_rna = sc.read(output_dir / "EEC_multiome_rna_matched.h5ad")
adata_atac = sc.read(output_dir / "EEC_multiome_atac_matched.h5ad")

In [ ]:
# Subset adata objects
keep_cells = list(adata_atac.obs_names)
for i, ad in enumerate(adatas):
    adatas[i] = ad[ad.obs_names.isin(keep_cells)].copy()

In [ ]:
# Concatenate all anndata objects
adata_atac_raw = adatas[0].concatenate(adatas[1:], batch_key="sample", batch_categories=basenames)

In [ ]:
adata_atac.obsm['fragment_paired'] = adata_atac_raw.obsm['fragment_paired']

In [ ]:
adata_atac.obs[["n_fragment", "frac_dup", "frac_mito", "sample"]] = adata_atac_raw.obs[["n_fragment", "frac_dup", "frac_mito", "sample"]]

In [ ]:
adata_atac.obs["final_annotation"] = adata_rna.obs["final_annotation"]
adata_atac.obs["final_annotation"] = adata_atac.obs["final_annotation"].str.replace("/", "_")

In [ ]:
adata_atac.write(output_dir / "EEC_multiome_atac_matched_raw.h5ad")

In [ ]:
s = pd.Series(genome.chrom_sizes, dtype="UInt64")

df = (
    s.rename_axis("reference_seq_name")
     .reset_index(name="reference_seq_length")
)

df["reference_seq_length"] = df["reference_seq_length"].astype("UInt64")

In [ ]:
adata_atac.uns["reference_sequences"] = df

In [ ]:
snap.ex.export_coverage(adata_atac, groupby='final_annotation', out_dir=output_dir, prefix='Intestine_EEC_Multiome_')